# 第82章 在线零售用户消费与RFM

<!-- module-learning-arc:start -->
> **综合项目 模块主线｜第 1 / 4 步：从交易明细理解客户价值**
>
> **持续应用背景：** 进入数据分析决策实验室：连续处理客户价值、物流履约、供需调度和营销资源四类问题，训练从业务问题到行动建议的迁移能力。
>
> **承接上一阶段：** 模块入门与应用任务  →  **本章任务：** 在线零售用户消费与RFM  →  **下一步：** Olist电商物流履约分析
>
> **大作业连接：** 本章练习将成为《跨模块业务决策项目》的一部分，最终需要把前四个项目形成的方法迁移为项目提案、最短充分证据链和决策备忘录。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：一间网店每天涌进来大量订单，光看"总销售额"很难知道这些钱是从哪些客户手里赚来的、又有哪些客户正在悄悄流失。RFM 把每位客户按"最近多久没买、一共买了几次、总共花了多少钱"拆成三个维度，一眼就能分清高价值大户和沉默的捡漏用户。本章就用一份真实的英国在线零售交易样本，带你从订单明细一路算到可以落地的运营客群。


## 本章目标

学完本章，你将能够：

- **理解**：理解「在线零售用户消费与RFM」的核心概念、适用场景与关键口径。
- **操作**：能按本章步骤写出可复现的实现，并读懂输出/结果。
- **迁移**：能用本章方法处理一份新数据，独立完成同类任务并给出结论。


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| InvoiceNo | 发票号 | 以 C 开头通常为取消单 |
| StockCode | 商品编码 | 商品主键 |
| Description | 商品描述 | 存在缺失 |
| Quantity | 数量 | 负数通常表示退货 |
| InvoiceDate | 交易时间 | 英国当地时间 |
| UnitPrice | 单价 | 英镑 |
| CustomerID | 客户编号 | 部分缺失 |
| Country | 客户国家 | 订单归属地 |

## 82.2 数据质量检查清单

- 发票行是否重复
- 取消单、负数量和非正单价占比
- CustomerID 与 Description 缺失率
- 交易日期范围与异常时间
- KPI 是否仅基于有效正向销售


## 项目任务

1. 读取数据并建立质量基线
2. 定义有效销售口径并构造收入
3. 计算月度与总体 KPI
4. 分析商品和国家贡献集中度
5. 构建 RFM 并划分运营客群
6. 输出行动建议与限制


## 项目阶段速查

先看每个阶段要做什么、留下什么证据，再按任务顺序运行项目代码。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 加载与质量审计 | `pd.read_csv()`、`pd.Series()`、`df.duplicated()`、`str.startswith()` | 先保留原始问题并量化，避免清洗后无法解释样本变化。 | 发票行是否重复 |
| 2. 清洗口径与经营KPI | `df.drop_duplicates()`、`str.startswith()`、`pd.Series()`、`sales.revenue.sum()` | 有效销售排除重复、取消、退货、非正价格；客户分析再要求客户编号非空。 | 取消单、负数量和非正单价占比 |
| 3. 月度趋势与集中度 | `sales.InvoiceDate.dt.to_period()`、`sales.groupby()`、`revenue.sum()`、`np.ceil()` | 趋势用于发现变化，Pareto 指标用于判断经营是否依赖少数商品或市场。 | CustomerID 与 Description 缺失率 |
| 4. RFM客户分层 | `sales.dropna()`、`customer_sales.InvoiceDate.max()`、`pd.Timedelta()`、`customer_sales.groupby()` | Recency 以数据末日次日为观察点；Frequency 使用不同发票数，Monetary 使用有效销售收入。 | 交易日期范围与异常时间 |
| 5. 决策摘要 | `segment.revenue.idxmax()`、`.sum()`、`int()`、`sum()` | 把指标转换成具体动作，同时保留抽样、缺失客户与退货口径的限制。 | KPI 是否仅基于有效正向销售 |


## 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 82.6 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 82.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 82.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 加载与质量审计

先保留原始问题并量化，避免清洗后无法解释样本变化。


<!-- math-foundation:chapter-82 -->
### 数学推导｜RFM 的三个客户价值维度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把交易明细聚合到客户。** 对客户 $i$ 的交易集合 $J_i$，频次 $F_i=|J_i|$，金额 $M_i=\sum_{j\in J_i}amount_j$。

**第 2 步｜用统一参考日计算最近度。** 最后交易日 $t_i^{last}=\max_{j\in J_i}t_j$，所以 $R_i=t_{ref}-t_i^{last}$。

**第 3 步｜方向统一后再评分。** 因为较小的 $R$ 更好，而较大的 $F$、$M$ 更好，可把分箱分数写成

$$
Score_i=w_Rs_R(-R_i)+w_Fs_F(F_i)+w_Ms_M(M_i)
$$

负号只表示最近度方向相反，不表示日期本身为负。

**把上面的关系收束为本章计算式：**

$$
R_i=t_{ref}-t_i^{last},\qquad F_i=N_i,\qquad M_i=\sum_{j\in i}amount_j
$$

**符号解释：** $R$ 是距最近消费的时间，$F$ 是消费频次，$M$ 是消费金额。

**代码对应：** 先聚合到客户粒度，再分别计算 R、F、M 并记录参考日期。

**使用边界：** 评分分箱依赖样本分布，跨时间或跨市场比较时必须重新校准。


In [ ]:
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import pandas as pd
import numpy as np

# 中文字体支持：自动选用可用的中文字体，避免图表中文显示为方框
if os.path.exists("/tmp/NotoSansSC-Regular.otf"):
    fm.fontManager.addfont("/tmp/NotoSansSC-Regular.otf")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"]
)
audit = pd.Series(
    {
        "行数": len(df),
        "重复行": df.duplicated().sum(),
        "客户缺失": df["CustomerID"].isna().sum(),
        "取消单": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
        "数量非正": (df["Quantity"] <= 0).sum(),
        "单价非正": (df["UnitPrice"] <= 0).sum(),
    }
)
print(audit.to_string())
print("日期:", df.InvoiceDate.min(), "至", df.InvoiceDate.max())
print(df.head())


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上手数据的第二件事，是弄清"这个市场到底有多重要"。上面已经加载了 `df` 并打印了行数、重复行、客户缺失等审计指标，现在请你再加一个口径：统计 `df` 中来自国家 `"United Kingdom"`（UK，本数据集最主流的市场）的订单明细行数 `uk_n`，并计算它占全部订单明细的比例 `uk_share`。在两个 `___` 处填好代码再运行，观察这个市场的集中程度。


In [ ]:
try:
    # 请在下方填写代码
    # 任务：统计 UK 市场的明细行数及其占比
    #   1) uk_n      -> 来自国家为 "United Kingdom" 的明细行数
    #      例：df["Country"].eq("United Kingdom").sum()
    #   2) uk_share  -> uk_n 占全部明细的比例（0~1 小数）
    #      例：uk_n / len(df)
    # 提示：df 已在上方单元格加载；布尔比较后用 .sum() 计数，分母用 len(df)。

    uk_n = None  # 请填写：UK 的明细行数
    uk_share = None  # 请填写：uk_n / len(df)

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 清洗口径与经营KPI

有效销售排除重复、取消、退货、非正价格；客户分析再要求客户编号非空。


In [ ]:
sales = df.drop_duplicates().copy()
valid = (
    (~sales["InvoiceNo"].astype(str).str.startswith("C"))
    & (sales["Quantity"] > 0)
    & (sales["UnitPrice"] > 0)
)
sales = sales.loc[valid].copy()
sales["revenue"] = sales["Quantity"] * sales["UnitPrice"]
kpi = pd.Series(
    {
        "有效明细": len(sales),
        "收入(GBP)": sales.revenue.sum(),
        "发票数": sales.InvoiceNo.nunique(),
        "有ID客户数": sales.CustomerID.nunique(),
        "客单价": sales.groupby("InvoiceNo").revenue.sum().mean(),
    }
)
print(kpi.round(2).to_string())
print(f"有效明细保留率: {len(sales)/len(df):.1%}")


## 月度趋势与集中度

趋势用于发现变化，Pareto 指标用于判断经营是否依赖少数商品或市场。


In [ ]:
# 中文字体支持：避免图表中文显示为方框
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

sales["month"] = sales.InvoiceDate.dt.to_period("M").astype(str)
monthly = (
    sales.groupby("month")
    .agg({"revenue": "sum", "InvoiceNo": "nunique"})
    .rename(columns={"InvoiceNo": "invoices"})
)
product = sales.groupby("StockCode").revenue.sum().sort_values(ascending=False)
country = sales.groupby("Country").revenue.sum().sort_values(ascending=False)
top20_n = max(1, int(np.ceil(len(product) * 0.2)))
print(monthly.round(0))
print(f"前20%商品收入贡献: {product.head(top20_n).sum()/product.sum():.1%}")
print(
    "主要国家贡献:\n",
    (country.head(8) / country.sum()).map(lambda x: f"{x:.1%}"),
)
plt.figure(figsize=(9, 4))
plt.plot(monthly.index.astype(str), monthly.revenue.values, marker="o")
plt.title("月度有效销售收入（GBP）")
plt.ylabel("GBP")
plt.tight_layout()
plt.show()


## RFM客户分层

Recency 以数据末日次日为观察点；Frequency 使用不同发票数，Monetary 使用有效销售收入。


In [ ]:
customer_sales = sales.dropna(subset=["CustomerID"])
snapshot = customer_sales.InvoiceDate.max().normalize() + pd.Timedelta(days=1)
rfm = (
    customer_sales.groupby("CustomerID")
    .agg(
        {
            "InvoiceDate": lambda x: (snapshot - x.max().normalize()).days,
            "InvoiceNo": "nunique",
            "revenue": "sum",
        }
    )
    .rename(
        columns={
            "InvoiceDate": "recency",
            "InvoiceNo": "frequency",
            "revenue": "monetary",
        }
    )
)
rfm["r_score"] = pd.qcut(
    rfm.recency.rank(method="first"), 4, labels=[4, 3, 2, 1]
).astype(int)
rfm["f_score"] = pd.qcut(
    rfm.frequency.rank(method="first"), 4, labels=[1, 2, 3, 4]
).astype(int)
rfm["m_score"] = pd.qcut(
    rfm.monetary.rank(method="first"), 4, labels=[1, 2, 3, 4]
).astype(int)
rfm["segment"] = np.select(
    [
        (rfm.r_score >= 3) & (rfm.f_score >= 3),
        (rfm.r_score <= 2) & (rfm.f_score >= 3),
        (rfm.r_score >= 3) & (rfm.f_score <= 2),
    ],
    ["高价值活跃", "高价值待唤回", "新近低频"],
    default="一般/沉睡",
)
segment = rfm.groupby("segment").agg(
    {"monetary": ["size", "sum"], "recency": ["median"]}
)
segment.columns = ["customers", "revenue", "median_recency"]
segment["revenue_share"] = segment.revenue / segment.revenue.sum()
print(segment.sort_values("revenue", ascending=False).round(2))


## 决策摘要

把指标转换成具体动作，同时保留抽样、缺失客户与退货口径的限制。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    best = segment.revenue.idxmax()
    risk_n = int((rfm.segment == "高价值待唤回").sum())
    print("经营建议")
    print(f"1. 收入最高客群为「{best}」，优先设计分层权益而非全量促销。")
    print(
        f"2. 对 {risk_n} 位高价值待唤回客户做小规模召回测试，并设置增量评估。"
    )
    print("3. 对高贡献商品建立缺货与退货监控，避免集中度转化为供应风险。")
    print(
        "限制：课程数据是原始数据的固定20万行样本；CustomerID缺失交易不能进入RFM；本分析描述关联，不证明营销动作的因果效果。"
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

**易错点 1**：R（最近一次消费）的参考日不统一——有人用"距今"，有人用"数据截止日"，分层结果不可比；先定死参考日再计算。

**易错点 2**：退货单（负数 Quantity）直接参与金额求和会把销售额算低；先定义"有效销售"口径（过滤退货或单独统计）。

**易错点 3**：CustomerID 缺失的行不是"零消费客户"，而是无法归属的记录；丢弃前先统计占比，避免样本偏差。

**易错点 4**：金额单位混用（英镑当人民币）会让 RFM 的 M 失真；聚合前先确认并统一单位。

**易错点 5**：分层边界（如四分位）随样本变化，报告要写清分位口径，否则他人复现不出同一分层。


## 结论与表达

- 清洗口径直接决定收入和客群结论，必须与结果一同交付。
- RFM 是运营排序工具，不是客户终身价值或因果响应模型。
- 集中度既意味着重点，也意味着供应与市场风险。


## 项目验收清单

- 能解释有效销售口径
- 能复算 KPI 与 Pareto 贡献
- 能说明 RFM 三指标观察窗口
- 建议包含目标客群、动作、指标和限制

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 小结

使用 UCI Online Retail 20 万行公开交易样本，完成用户消费口径清洗、经营 KPI、消费集中度与 RFM 客群运营分析。


### 你已经完成

- 建立可复核的交易清洗口径
- 从交易明细构造经营 KPI
- 识别商品与国家贡献集中度
- 用 RFM 形成可行动的客户分层
- 把结果写成决策建议而非图表描述


### 质量与结论提醒

- 发票行是否重复
- 取消单、负数量和非正单价占比
- CustomerID 与 Description 缺失率
- 清洗口径直接决定收入和客群结论，必须与结果一同交付。
- RFM 是运营排序工具，不是客户终身价值或因果响应模型。
- 集中度既意味着重点，也意味着供应与市场风险。


### 项目交付检查

- [ ] 能解释有效销售口径
- [ ] 能复算 KPI 与 Pareto 贡献
- [ ] 能说明 RFM 三指标观察窗口
- [ ] 建议包含目标客群、动作、指标和限制


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：统计 UK 市场的明细行数及其占比
uk_n = df["Country"].eq("United Kingdom").sum()
uk_share = uk_n / len(df)

print(f"UK 明细行数：{uk_n}")
print(f"UK 占比：{uk_share:.1%}")
